# İstatistiksel Testler

Bu notebook araçlar arasındaki performans farklarının istatistiksel anlamlılığını test eder.

In [ ]:
import json
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')

In [ ]:
# Deney sonuçlarını yükle
results_dir = Path('../../data/results')
result_files = sorted(results_dir.glob('exp_*.json'))

if result_files:
    with open(result_files[-1], 'r', encoding='utf-8') as f:
        experiment = json.load(f)
    
    # DataFrame oluştur
    results_data = []
    for result in experiment['results']:
        for tool, metrics in result.get('metrics', {}).items():
            if metrics:
                results_data.append({
                    'tool': tool,
                    'category': result['category'],
                    'bleu': metrics.get('bleu'),
                    'meteor': metrics.get('meteor'),
                    'ter': metrics.get('ter'),
                    'chrf': metrics.get('chrf')
                })
    
    df = pd.DataFrame(results_data)
    print(f"Toplam çeviri: {len(df)}")
    print(f"Araçlar: {df['tool'].unique()}")

## Paired t-test (Araçlar Arası)

In [ ]:
# Araçlar arasında paired t-test
tools = df['tool'].unique()
metric = 'bleu'

print(f"\nPaired t-test Sonuçları ({metric.upper()}):")
print("="*60)

for i, tool1 in enumerate(tools):
    for tool2 in tools[i+1:]:
        scores1 = df[df['tool'] == tool1][metric].dropna()
        scores2 = df[df['tool'] == tool2][metric].dropna()
        
        # Aynı uzunlukta olmalı (paired test için)
        min_len = min(len(scores1), len(scores2))
        scores1 = scores1[:min_len]
        scores2 = scores2[:min_len]
        
        t_stat, p_value = stats.ttest_rel(scores1, scores2)
        
        mean_diff = scores1.mean() - scores2.mean()
        
        significance = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
        
        print(f"\n{tool1} vs {tool2}:")
        print(f"  t-statistic: {t_stat:.4f}")
        print(f"  p-value: {p_value:.6f} {significance}")
        print(f"  Mean difference: {mean_diff:.4f}")
        print(f"  {tool1} mean: {scores1.mean():.4f}")
        print(f"  {tool2} mean: {scores2.mean():.4f}")

## ANOVA (Çoklu Grup Karşılaştırması)

In [ ]:
# One-way ANOVA
groups = [df[df['tool'] == tool]['bleu'].dropna() for tool in tools]

f_stat, p_value = stats.f_oneway(*groups)

print("\nOne-way ANOVA (BLEU):")
print("="*60)
print(f"F-statistic: {f_stat:.4f}")
print(f"p-value: {p_value:.6f}")

if p_value < 0.05:
    print("\n✓ Araçlar arasında istatistiksel olarak anlamlı fark var (p < 0.05)")
else:
    print("\n✗ Araçlar arasında anlamlı fark yok (p >= 0.05)")

## Effect Size (Cohen's d)

In [ ]:
def cohens_d(group1, group2):
    """Cohen's d effect size hesapla"""
    n1, n2 = len(group1), len(group2)
    var1, var2 = group1.var(), group2.var()
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (group1.mean() - group2.mean()) / pooled_std

print("\nEffect Size (Cohen's d) - BLEU:")
print("="*60)
print("Yorumlama: |d| < 0.2 (küçük), 0.2-0.8 (orta), > 0.8 (büyük)\n")

for i, tool1 in enumerate(tools):
    for tool2 in tools[i+1:]:
        scores1 = df[df['tool'] == tool1]['bleu'].dropna()
        scores2 = df[df['tool'] == tool2]['bleu'].dropna()
        
        d = cohens_d(scores1, scores2)
        
        magnitude = "büyük" if abs(d) > 0.8 else "orta" if abs(d) > 0.2 else "küçük"
        
        print(f"{tool1} vs {tool2}: d = {d:.4f} ({magnitude} etki)")

## Kategori Bazlı İstatistiksel Testler

In [ ]:
# Her kategori için ayrı ANOVA
for category in df['category'].unique():
    print(f"\n{'='*60}")
    print(f"Kategori: {category.upper()}")
    print("="*60)
    
    cat_df = df[df['category'] == category]
    groups = [cat_df[cat_df['tool'] == tool]['bleu'].dropna() for tool in tools]
    
    f_stat, p_value = stats.f_oneway(*groups)
    
    print(f"F-statistic: {f_stat:.4f}")
    print(f"p-value: {p_value:.6f}")
    
    if p_value < 0.05:
        print("✓ Anlamlı fark var")
    else:
        print("✗ Anlamlı fark yok")
    
    # Ortalama skorlar
    print("\nOrtalama BLEU skorları:")
    for tool in tools:
        mean_score = cat_df[cat_df['tool'] == tool]['bleu'].mean()
        print(f"  {tool}: {mean_score:.4f}")

## Sonuç

İstatistiksel testler tamamlandı. Bulgular makaleye eklenebilir.